In [ ]:
class CNN1D(nn.Module):
    def __init__(self, in_ch, n_cities, channels, n_layers, kernel, head_hidden, dropout, emb_dim=8, out_dim=3):
        super().__init__()
        blocks, c = [], in_ch
        for _ in range(n_layers):
            blocks += [nn.Conv1d(c, channels, kernel, padding=kernel // 2), nn.BatchNorm1d(channels), nn.ReLU()]
            c = channels
        self.conv = nn.Sequential(*blocks)
        self.emb = nn.Embedding(n_cities, emb_dim)
        self.head = nn.Sequential(nn.Linear(2 * channels + emb_dim, head_hidden), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(head_hidden, out_dim))

    def forward(self, x, city):
        z = self.conv(x)
        z = torch.cat([z.mean(dim=2), z.amax(dim=2)], dim=1)
        return self.head(torch.cat([z, self.emb(city)], dim=1))


